# 07 · Duplicados

Este notebook implementa y evalúa una regla de detección de duplicados entre productos del catálogo.

Objetivos:
- Cargar los datasets de desarrollo y evaluación.
- Generar embeddings de productos usando E5-small.
- Calibrar un umbral de similitud que maximice F1.
- Evaluar la regla con métricas: Precision, Recall y F1.
- Aplicar la regla a `altas_evaluacion.csv`.
- Generar el archivo `resultados_duplicados.csv`.

Este notebook utiliza:
- `duplicates.py`
- `embeddings.py`
- `utils.py`


#### Importar librerías

In [1]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [2]:
import pandas as pd
import numpy as np

from src.utils import safe_read_csv, log_section
from src.duplicates import (
    calibrate_threshold,
    evaluate_rule,
    apply_rule_to_evaluation
)


#### Cargar datasets de duplicados

In [3]:
log_section("Cargar datasets de duplicados")

df_dev = safe_read_csv("../data/altas_desarrollo.csv")
df_eval = safe_read_csv("../data/altas_evaluacion.csv")

df_dev.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar datasets de duplicados
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/altas_desarrollo.csv (14 filas)
[AURUM] [CSV] Cargado: ../data/altas_evaluacion.csv (14 filas)


,incoming_id,title,brand,color,text,is_duplicate,reference_product_id
0,DEV-DUP-001,NIKE Legasee Legging Swoosh Pantalones Deporti...,NIKE,Negro (Black/White 011),NIKE Legasee Legging Swoosh Pantalones Deporti...,True,B000G3T55M
1,DEV-DUP-002,"Interruptor Universal Inteligente con Wi-Fi, c...",meross,Blanco,"Interruptor Universal Inteligente con Wi-Fi, c...",True,B07NV4L2W5
2,DEV-DUP-003,gel de contacto 250g. | mejora la conductivida...,axion,NaN,gel de contacto 250g. | mejora la conductivida...,True,B00BEFAR80
3,DEV-DUP-004,"Teka Campana Extractora, Touch Control y Motor...",Teka,"Negro, Acero Inoxidable","Teka Campana Extractora, Touch Control y Motor...",True,B076HKFZ8N
4,DEV-DUP-005,[Casio] de CASIO Frogman 35 Aniversario océano...,G-Shock,NaN,[Casio] de CASIO Frogman 35 Aniversario océano...,True,B07JYHSK27


#### Dataset de desarrollo

Este dataset contiene altas de productos que llegan al sistema y deben evaluarse para determinar si son duplicados de un producto existente en el catálogo.

Cada fila representa una alta individual, con las siguientes columnas:

- incoming_id = Identificador único de la alta.
- title = Título del producto nuevo.
- brand = Marca del producto nuevo.
- color = Color del producto nuevo.
- text = Descripción del producto nuevo.
- is_duplicate  
    Etiqueta binaria:
    - True → la alta es duplicado de un producto existente
    - False → la alta es un producto nuevo válido
- reference_product_id  
    Si is_duplicate=True, este campo indica el product_id del catálogo al que corresponde el duplicado.
    Si is_duplicate=False, está vacío.

Este dataset se utiliza para calibrar y evaluar la regla de detección de duplicados, comparando:
- La similitud entre la alta y el producto más parecido del catálogo.
- La etiqueta real (is_duplicate).
- El umbral de decisión que maximiza F1.


#### Calibrar el umbral óptimo

In [4]:
log_section("Calibrar umbral óptimo")

threshold = calibrate_threshold("../data/altas_desarrollo.csv", model_name="e5_small")
threshold


[AURUM] 
[AURUM] ============================================================
[AURUM] Calibrar umbral óptimo
[AURUM] ============================================================


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[AURUM] [EMBEDDINGS] Modelo cargado: e5_small (intfloat/multilingual-e5-small)
[AURUM] [DUPLICATES] Umbral óptimo: 0.9071 (F1=1.0000)


np.float64(0.9071289558662414)

#### Interpretación del umbral

El umbral encontrado es el valor de similitud (dot product) que maximiza F1.

Ejemplo:
- Si el umbral es **0.82**, entonces:
  - similitud ≥ 0.82 → duplicado
  - similitud < 0.82 → no duplicado

Este umbral se obtiene probando cientos de thresholds entre percentiles 50 y 99.


#### Evaluación de la regla en desarrollo

In [5]:
log_section("Evaluación de la regla")

precision, recall, f1 = evaluate_rule("../data/altas_desarrollo.csv", threshold, model_name="e5_small")

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


[AURUM] 
[AURUM] ============================================================
[AURUM] Evaluación de la regla
[AURUM] ============================================================
[AURUM] [DUPLICATES] Precision=1.0000, Recall=1.0000, F1=1.0000
Precision: 0.9999999998571428
Recall: 0.9999999998571428
F1: 0.9999999993571429


#### Interpretación de métricas

### Precision
Proporción de predicciones positivas que son correctas.
- Alta precision → pocos falsos duplicados.

### Recall
Proporción de duplicados reales que detectamos.
- Alto recall → pocos duplicados perdidos.

### F1
Media armónica entre precision y recall.
- F1 alto → buen equilibrio.

Conclusión:
- La regla calibrada ofrece un rendimiento sólido.
- El umbral es adecuado para aplicar en evaluación.


#### Aplicar la regla a evaluación

In [6]:
log_section("Aplicar regla a evaluación")

apply_rule_to_evaluation("../data/altas_evaluacion.csv", threshold, model_name="e5_small")

print("Archivo resultados_duplicados.csv generado en ../results/")


[AURUM] 
[AURUM] ============================================================
[AURUM] Aplicar regla a evaluación
[AURUM] ============================================================
[AURUM] [DUPLICATES] Archivo resultados_duplicados.csv generado correctamente.
Archivo resultados_duplicados.csv generado en ../results/


#### Ver el archivo generado

In [7]:
log_section("Ver archivo generado")

df_out = safe_read_csv("../results/resultados_duplicados.csv")
df_out.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Ver archivo generado
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../results/resultados_duplicados.csv (14 filas)


,incoming_id,predicted_duplicate,matched_product_id,score
0,EVAL-DUP-001,True,B081JP8CC6,0.945576
1,EVAL-DUP-002,True,B07GWRF23V,0.984829
2,EVAL-DUP-003,True,B07S7B3SN2,0.959296
3,EVAL-DUP-004,True,8417441271,0.923008
4,EVAL-DUP-005,True,B00JOH9FRO,0.949699


#### Conclusiones

Este notebook demuestra:

### ✔ Calibración del umbral
- Se obtiene el threshold óptimo mediante maximización de F1.
- El modelo E5-small produce similitudes coherentes.

### ✔ Evaluación con métricas
- Precision, Recall y F1 permiten justificar la calidad de la regla.
- La regla es simple pero efectiva.

### ✔ Aplicación a evaluación
- Se genera el archivo `resultados_duplicados.csv`.
- Este archivo es uno de los entregables obligatorios del proyecto.

### ✔ Robustez del sistema
- La regla funciona sobre texto real del catálogo.
- El pipeline de embeddings y similitud está correctamente integrado.

En el siguiente notebook calcularemos **métricas de ranking** y **fidelidad ANN**.
